In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import chardet
import uuid

In [2]:
def get_last_page_nr(url: str) -> int:
    '''Returns the total number of pages

    Parameters:
    url (str): the url of the page where you want to extract from.

    Returns:
    int: the number of the last page'''

    response = requests.get(url)
    soup = BeautifulSoup(response.content, features="html.parser")

    last_page_number = soup.find("a", class_="last-page").get_text(strip=False)
    return int(last_page_number)

url = 'https://www.imoti.net/bg/obiavi/r/prodava/sofia/?page=1&sid=gSoWpd'
last_page_number = get_last_page_nr(url)


In [3]:
url = 'https://www.imoti.net/bg/obiavi/r/prodava/sofia/?page=1&sid=gSoWpd'
response = requests.get(url)
# print(response.text)

In [4]:
import hashlib

def generate_ad_id(title, url):
    '''Generate unique customized ID of the ad based on stable fields.
    
    Parameters:
    title (str): ad title
    url (str): link to the ad
    
    Returns:
    str'''
    key = f"{title}|{url}"     # choose stable fields
    return hashlib.sha256(key.encode()).hexdigest()

In [5]:
# Detect the encoding of the content
detected_encoding = chardet.detect(response.content)['encoding']

# Decode the content using the detected encoding
if detected_encoding:
    content = response.content.decode(detected_encoding)
else:
    # Fallback to UTF-8 if encoding detection fails
    content = response.content.decode('utf-8', errors='ignore')

soup = BeautifulSoup(content, "html.parser")
box_items = soup.find_all('li', class_ = 'clearfix')
# print(box_items)

prefix = "https://www.imoti.net"
real_estate_dict = {}
for real_estate in box_items:
    link_to_ad = real_estate.find('a').get('href')
    ad_title = real_estate.find('img').get('alt')

    # generate unique ID of the ad based on the title & url
    ad_id = generate_ad_id(
        title=ad_title,
        url=link_to_ad
        )
    
    # add a new element to the dict
    real_estate_dict[ad_id] = {'ad_title': ad_title, 'link_to_ad': prefix + link_to_ad}

real_estate_dict


{'0aa1295eae5aaae8932ee46dd5f94977fb36d4e41646c6fffee2ba7bb80dadaf': {'ad_title': 'Имот - продава Едностаен апартамент, в София, Овча Купел',
  'link_to_ad': 'https://www.imoti.net/bg/obiava/prodava/sofia/ovcha-kupel/ednostaen/6176507/?sid=gSoWpd&page=1'},
 'a34187e9c76a3dd8c10112fc7020c14c65446eda8fef9225df386dc60ddfc5af': {'ad_title': 'Имот - продава Едностаен апартамент, в София, Фондови Жилища',
  'link_to_ad': 'https://www.imoti.net/bg/obiava/prodava/sofia/fondovi-jilishta/ednostaen/5883830/?sid=gSoWpd&page=1'},
 '30e1ddf1b15033c52f485def03d1a106fd791016e035686d23a6d3422fe96776': {'ad_title': 'Имот - продава Двустаен апартамент, в София, Левски',
  'link_to_ad': 'https://www.imoti.net/bg/obiava/prodava/sofia/sofija-levski/dvustaen/6176496/?sid=gSoWpd&page=1'},
 'd04a6e7b3928942e31c0b6ebdfebd6e2b12066410c013e343fcb9767aed88f4c': {'ad_title': 'Имот - продава Двустаен апартамент, в София, Карпузица',
  'link_to_ad': 'https://www.imoti.net/bg/obiava/prodava--v-stroej/sofia/karpuzica/d

In [6]:
real_estate_dict["0aa1295eae5aaae8932ee46dd5f94977fb36d4e41646c6fffee2ba7bb80dadaf"]

{'ad_title': 'Имот - продава Едностаен апартамент, в София, Овча Купел',
 'link_to_ad': 'https://www.imoti.net/bg/obiava/prodava/sofia/ovcha-kupel/ednostaen/6176507/?sid=gSoWpd&page=1'}

In [7]:
# experiment with the page of 1 ad
example_page = real_estate_dict["0aa1295eae5aaae8932ee46dd5f94977fb36d4e41646c6fffee2ba7bb80dadaf"]
example_url = example_page['link_to_ad']
print(example_url)

https://www.imoti.net/bg/obiava/prodava/sofia/ovcha-kupel/ednostaen/6176507/?sid=gSoWpd&page=1


In [8]:
response_specific_page = requests.get(example_url)

# Detect the encoding of the content
detected_encoding = chardet.detect(response_specific_page.content)['encoding']

# Decode the content using the detected encoding
if detected_encoding:
    content = response_specific_page.content.decode(detected_encoding)
else:
    # Fallback to UTF-8 if encoding detection fails
    content = response_specific_page.content.decode('utf-8', errors='ignore')

In [9]:
# use the link to fetch the additional information
soup_specific_page = BeautifulSoup(content, "html.parser")
# print(soup_specific_page)

# grab the sidebar (single element)
aside_info = soup_specific_page.find("aside", class_="info-sidebar")
print(aside_info)

# find simple-table divs and filter by whether they contain td or p
simple_tables = aside_info.find_all("div", class_="simple-table") if aside_info else []
tables_with_td = [tbl for tbl in simple_tables if tbl.find("td")]
tables_with_p = [tbl for tbl in simple_tables if tbl.find("p")]

# if you want the first matching table only
table_with_td_info = tables_with_td[0] if tables_with_td else None
table_with_p_info = tables_with_p[0] if tables_with_p else None

# description lives on the specific page
description = soup_specific_page.find("div", class_="text")

<aside class="info-sidebar">
<section class="info-sidebar-box clearfix">
<header class="blue-header clearfix">
<div class="big-price" style="width: 100%;">
<strong>92 500 €</strong>
</div>
<div class="big-price">
<strong>180 914 BGN</strong>
</div>
</header>
<hr class="solid"/>
<div class="simple-table">
<table>
<thead>
<tr>
<th colspan="3">Данни за имота:</th>
</tr>
</thead>
<tbody>
<tr>
<td>Цена на м<sup>2</sup>/EUR/:</td>
<td>1 888</td>
</tr>
<tr>
<td>Цена на м<sup>2</sup>/BGN/:</td>
<td>3 692.13</td>
</tr>
<tr>
<td>Квадратура /м<sup>2</sup>/:</td>
<td>49</td>
</tr>
<tr>
<td>Етаж :</td>
<td>1 от 6</td>
</tr>
<tr>
<td>Строителство:</td>
<td>Тухла</td>
</tr>
<tr>
<td>Акт 16:</td>
<td>Да</td>
</tr>
<tr>
<td>Енергиен клас:</td>
<td>N/A</td>
</tr>
<tr>
<td>Потребление:</td>
<td>
                            N/A
                                                    </td>
</tr>
</tbody>
</table>
</div>
<hr class="solid"/>
<ul class="side-links">
<li><a class="ext-link" href="/bg/sredni-ceni/?

In [10]:
table_with_p_info

<div class="simple-table">
<p><strong>Дължи се комисион на брокера:</strong>
<span class="fr"><strong>Да</strong></span>
</p>
<p><strong>Бележки:</strong>
                Посочената цена не включва местни данъци и такси.
                </p>
</div>

In [11]:
# extract text from <p> tags inside the simple-table
p_texts = [p.get_text(" ", strip=True) for p in table_with_p_info.find_all("p")] if table_with_p_info else []
p_texts

# optional: split "key: value" style lines into a dict
p_data = {}
for text in p_texts:
    if ":" in text:
        key, value = text.split(":", 1)
        p_data[key.strip()] = value.strip()
p_data

{'Дължи се комисион на брокера': 'Да',
 'Бележки': 'Посочената цена не включва местни данъци и такси.'}

In [12]:
data={}
for row in table_with_td_info.find_all("tr"):
    cols = row.find_all("td")
    if len(cols) >= 2:
        key = cols[0].get_text(strip=True)
        value = cols[1].get_text(strip=True)
        data[key] = value

# for row in table_with_td_info.find_all("tr"):
#     cols = row.find_all("td")
#     if len(cols) >= 2:
#         key = cols[0].get_text(strip=True)
#         value = cols[1].get_text(strip=True)
#         data[key] = value
data

{'Цена на м2/EUR/:': '1 888',
 'Цена на м2/BGN/:': '3 692.13',
 'Квадратура /м2/:': '49',
 'Етаж :': '1 от 6',
 'Строителство:': 'Тухла',
 'Акт 16:': 'Да',
 'Енергиен клас:': 'N/A',
 'Потребление:': 'N/A'}

In [14]:
# slice first 3 elements
sample_ad_items = dict(list(real_estate_dict.items())[:3])

for id, ad_item in sample_ad_items.items():
    print(10*"=" + f"Fetching the html for item {id}..." + 10*"=")
    item_response = requests.get(ad_item['link_to_ad'])
    item_soup = BeautifulSoup(item_response.text, "html.parser")
    
    print(10*"=" + "Parsing the HTML..." + 10*"=")
    # grab the sidebar (single element)
    aside_info = item_soup.find("aside", class_="info-sidebar")
    # print(aside_info)

    # find simple-table divs and filter by whether they contain td or p
    simple_tables = aside_info.find_all("div", class_="simple-table") if aside_info else []
    tables_with_td = [tbl for tbl in simple_tables if tbl.find("td")]
    tables_with_p = [tbl for tbl in simple_tables if tbl.find("p")]

    # if you want the first matching table only
    table_with_td_info = tables_with_td[0] if tables_with_td else None
    table_with_p_info = tables_with_p[0] if tables_with_p else None

    # description lives on the specific page
    description = item_soup.find("div", class_="text")
    description_text = description.get_text(" ", strip=True) if description else ""
    ad_item["description"] = description_text
    print(description_text)
    
    for row in table_with_td_info.find_all("tr") if table_with_td_info else []:
        cols = row.find_all("td")
        if len(cols) >= 2:
            key = cols[0].get_text(strip=True)
            value = cols[1].get_text(strip=True)
            ad_item[key] = value

    # extract text from <p> tags inside the simple-table
    p_texts = [p.get_text(" ", strip=True) for p in table_with_p_info.find_all("p")] if table_with_p_info else []
    p_texts

    # optional: split "key: value" style lines into a dict
    # p_data = {}
    for text in p_texts:
        if ":" in text:
            key, value = text.split(":", 1)
            ad_item[key.strip()] = value.strip()

    print(10*"=" + f"Finished with item {id}" + 10*"=")

print(sample_ad_items)

==========Fetching the html for item 0aa1295eae5aaae8932ee46dd5f94977fb36d4e41646c6fffee2ba7bb80dadaf...==========
==========Parsing the HTML...==========
ИМА ВЪЗМОЖНОСТ ЗА ЗАКУПУВАНЕ НА ПАРКОМЯСТО. Продава се просторен едностаен апартамент с площ 49 кв.м, разположен на 1-ви етаж от 6 в тухлена сграда с добре поддържани общи части. Имотът се намира в един от най-предпочитаните райони на София – кв. Овча купел, в близост до метростанция, градски транспорт, аптеки, магазини и основни пътни връзки, осигуряващи лесен и бърз достъп до всички части на града. Апартаментът се предлага на шпакловка и замазка, което дава възможност за довършване по индивидуален вкус. Сградата е нова, тухлена, с асансьор и контролиран достъп. Имотът е подходящ както за живеене, така и за инвестиция с цел отдаване под наем. За повече информация и за организиране на оглед не се колебайте да се свържете с нас. Ще се радваме да ви предоставим всички детайли и да уговорим удобен час за посещение на имота! Отговорен бр

In [6]:
# extract the extras section

url = "https://www.imoti.net/bg/obiava/prodava/sofia/drujba-1/tristaen/6213086/"
response = requests.get(url)

soup = BeautifulSoup(response.content, "html.parser")

result = soup.find("ul", class_="extras")
result

# result_dict = {}



<ul class="extras">
<li>ТЕЦ</li>
<li>Асансьор</li>
<li>Ново строителство</li>
<li>Паркомясто</li>
<li>До метростанция - до 10 мин.пеш</li>
</ul>

In [11]:
li_items = result.find_all('li')
result_dict = {"extras": "; ".join([li.get_text(strip=True) for li in li_items])}
result_dict

{'extras': 'ТЕЦ; Асансьор; Ново строителство; Паркомясто; До метростанция - до 10 мин.пеш'}